In [1]:
import duckdb
import pandas as pd
from pathlib import Path

PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"

# connect to an in-memory duckdb instance
con = duckdb.connect()

# register parquet files as views (no copy, queried lazily)
con.execute(f"""
    CREATE VIEW ratings AS SELECT * FROM read_parquet('{PARQUET_DIR}/ratings.parquet');
    CREATE VIEW movies  AS SELECT * FROM read_parquet('{PARQUET_DIR}/movies.parquet');
    CREATE VIEW tags    AS SELECT * FROM read_parquet('{PARQUET_DIR}/tags.parquet');
""")

# verify by listing tables and showing schema
print("registered views:")
print(con.execute("SHOW TABLES").fetchdf())

print("\nratings schema:")
print(con.execute("DESCRIBE ratings").fetchdf())

print("\nmovies schema:")
print(con.execute("DESCRIBE movies").fetchdf())

registered views:
      name
0   movies
1  ratings
2     tags

ratings schema:
  column_name column_type null   key default extra
0      userId      BIGINT  YES  None    None  None
1     movieId      BIGINT  YES  None    None  None
2      rating      DOUBLE  YES  None    None  None
3   timestamp      BIGINT  YES  None    None  None

movies schema:
  column_name column_type null   key default extra
0     movieId      BIGINT  YES  None    None  None
1       title     VARCHAR  YES  None    None  None
2      genres     VARCHAR  YES  None    None  None


In [2]:
# query 1: total counts
q = """
SELECT 
    COUNT(*) AS total_ratings,
    COUNT(DISTINCT userId) AS total_users,
    COUNT(DISTINCT movieId) AS total_movies,
    AVG(rating) AS mean_rating,
    MIN(rating) AS min_rating,
    MAX(rating) AS max_rating
FROM ratings
"""
print(con.execute(q).fetchdf())

   total_ratings  total_users  total_movies  mean_rating  min_rating  \
0       25000095       162541         59047     3.533854         0.5   

   max_rating  
0         5.0  


In [3]:
q = """
SELECT 
    rating,
    COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM ratings
GROUP BY rating
ORDER BY rating
"""
con.execute(q).fetchdf()

,rating,cnt,pct
0,0.5,393068,1.57
1,1.0,776815,3.11
2,1.5,399490,1.60
3,2.0,1640868,6.56
4,2.5,1262797,5.05
5,3.0,4896928,19.59
6,3.5,3177318,12.71
7,4.0,6639798,26.56
8,4.5,2200539,8.80
9,5.0,3612474,14.45


In [4]:
q = """
SELECT 
    userId,
    COUNT(*) AS num_ratings,
    AVG(rating) AS avg_rating,
    MIN(rating) AS lowest,
    MAX(rating) AS highest,
    COUNT(DISTINCT movieId) AS distinct_movies
FROM ratings
GROUP BY userId
ORDER BY num_ratings DESC
LIMIT 10
"""
con.execute(q).fetchdf()

,userId,num_ratings,avg_rating,lowest,highest,distinct_movies
0,72315,32202,3.080601,0.5,5.0,32202
1,80974,9178,3.280290,0.5,5.0,9178
2,137293,8913,3.184001,0.5,5.0,8913
3,33844,7919,2.580124,0.5,5.0,7919
4,20055,7488,3.208868,1.0,5.0,7488
5,109731,6647,2.816684,0.5,5.0,6647
6,92046,6564,3.475244,0.5,5.0,6564
7,49403,6553,1.522585,0.5,5.0,6553
8,30879,5693,2.876515,0.5,5.0,5693
9,115102,5649,2.462825,0.5,5.0,5649


In [5]:
q = """
SELECT 
    m.title,
    COUNT(*) AS num_ratings,
    ROUND(AVG(r.rating), 3) AS avg_rating,
    ROUND(STDDEV(r.rating), 3) AS rating_std
FROM ratings r
JOIN movies m ON r.movieId = m.movieId
GROUP BY m.movieId, m.title
ORDER BY num_ratings DESC
LIMIT 10
"""
con.execute(q).fetchdf()

,title,num_ratings,avg_rating,rating_std
0,Forrest Gump (1994),81491,4.048,0.939
1,"Shawshank Redemption, The (1994)",81482,4.414,0.760
2,Pulp Fiction (1994),79672,4.189,0.959
3,"Silence of the Lambs, The (1991)",74127,4.151,0.862
4,"Matrix, The (1999)",72674,4.154,0.913
5,Star Wars: Episode IV - A New Hope (1977),68717,4.120,0.982
6,Jurassic Park (1993),64144,3.679,0.940
7,Schindler's List (1993),60411,4.248,0.875
8,Braveheart (1995),59184,4.002,0.968
9,Fight Club (1999),58773,4.228,0.870


In [6]:
q = """
WITH movie_genre AS (
    SELECT 
        m.movieId,
        m.title,
        UNNEST(STRING_SPLIT(m.genres, '|')) AS genre,
        COUNT(*) OVER (PARTITION BY m.movieId) AS dummy
    FROM movies m
),
movie_stats AS (
    SELECT 
        mg.genre,
        mg.title,
        m.movieId,
        COUNT(*) AS num_ratings,
        AVG(r.rating) AS avg_rating
    FROM ratings r
    JOIN movies m ON r.movieId = m.movieId
    JOIN movie_genre mg ON mg.movieId = m.movieId
    WHERE mg.genre != '(no genres listed)'
    GROUP BY mg.genre, mg.title, m.movieId
),
ranked AS (
    SELECT 
        genre,
        title,
        num_ratings,
        ROUND(avg_rating, 2) AS avg_rating,
        ROW_NUMBER() OVER (PARTITION BY genre ORDER BY num_ratings DESC) AS rank_in_genre
    FROM movie_stats
)
SELECT genre, rank_in_genre, title, num_ratings, avg_rating
FROM ranked
WHERE rank_in_genre <= 5
ORDER BY genre, rank_in_genre
"""
con.execute(q).fetchdf()

,genre,rank_in_genre,title,num_ratings,avg_rating
0,Action,1,"Matrix, The (1999)",72674,4.15
1,Action,2,Star Wars: Episode IV - A New Hope (1977),68717,4.12
2,Action,3,Jurassic Park (1993),64144,3.68
3,Action,4,Braveheart (1995),59184,4.00
4,Action,5,Fight Club (1999),58773,4.23
...,...,...,...,...,...
90,Western,1,Dances with Wolves (1990),41615,3.73
91,Western,2,Back to the Future Part III (1990),21881,3.34
92,Western,3,Django Unchained (2012),20687,4.00
93,Western,4,"Good, the Bad and the Ugly, The (Buono, il bru...",18162,4.13


In [8]:
q = """
SELECT * FROM (
    (
        SELECT 'harsh' AS user_type, userId, COUNT(*) AS num_ratings, ROUND(AVG(rating), 2) AS avg_rating
        FROM ratings GROUP BY userId
        HAVING COUNT(*) >= 100
        ORDER BY avg_rating ASC LIMIT 10
    )
    UNION ALL
    (
        SELECT 'lenient' AS user_type, userId, COUNT(*) AS num_ratings, ROUND(AVG(rating), 2) AS avg_rating
        FROM ratings GROUP BY userId
        HAVING COUNT(*) >= 100
        ORDER BY avg_rating DESC LIMIT 10
    )
) ORDER BY user_type, avg_rating
"""
con.execute(q).fetchdf()

,user_type,userId,num_ratings,avg_rating
0,harsh,131800,293,0.51
1,harsh,88243,237,0.80
2,harsh,8901,257,0.84
3,harsh,13838,353,0.89
4,harsh,114172,163,0.98
5,harsh,106065,585,0.99
6,harsh,91064,105,1.01
7,harsh,45040,884,1.01
8,harsh,149767,872,1.03
9,harsh,110873,179,1.03


a simple aggregate-and-filter query revealed obvious data quality issues: user 75309 rated 5,525 movies all at 5.0 (std = 0), user 131800 rated 293 all near 0.5. these are likely bots or scripted accounts. before training, i'll filter users where stddev(rating) < 0.3 AND count >= 50. this kind of filtering is invisible if you only look at population-level stats; it surfaces only when you slice per-user.